In [0]:
# DataQuality

sales = spark.table(
    "workspace.olist_ecommerce.gold_sales_performance"
)

customers = spark.table(
    "workspace.olist_ecommerce.gold_customer_360"
)

products = spark.table(
    "workspace.olist_ecommerce.gold_product_performance"
)

sellers = spark.table(
    "workspace.olist_ecommerce.gold_seller_performance"
)

delivery = spark.table(
    "workspace.olist_ecommerce.gold_delivery_performance"
)

payments = spark.table(
    "workspace.olist_ecommerce.gold_payment_analysis"
)

reviews = spark.table(
    "workspace.olist_ecommerce.gold_review_analysis"
)

print("All Gold tables loaded successfully!")

All Gold tables loaded successfully!


In [0]:
from pyspark.sql import functions as F

quality_results = []

def check_nulls(df, table_name, columns):
    for column in columns:
        null_count = df.filter(
            F.col(column).isNull()
        ).count()

        quality_results.append({
            "table": table_name,
            "check": f"NULL check: {column}",
            "failed_records": null_count,
            "status": "PASS" if null_count == 0 else "FAIL"
        })


# Sales
check_nulls(
    sales,
    "gold_sales_performance",
    ["order_id", "customer_id", "order_status"]
)

# Customers
check_nulls(
    customers,
    "gold_customer_360",
    ["customer_id"]
)

# Products
check_nulls(
    products,
    "gold_product_performance",
    ["product_id"]
)

# Sellers
check_nulls(
    sellers,
    "gold_seller_performance",
    ["seller_id"]
)

# Delivery
check_nulls(
    delivery,
    "gold_delivery_performance",
    ["purchase_year", "purchase_month"]
)

# Payments
check_nulls(
    payments,
    "gold_payment_analysis",
    ["payment_type"]
)

# Reviews
check_nulls(
    reviews,
    "gold_review_analysis",
    ["review_score_int"]
)

print("Completeness checks completed.")

Completeness checks completed.


In [0]:
def check_duplicates(df, table_name, key_column):

    total_records = df.count()

    distinct_records = (
        df.select(key_column)
        .distinct()
        .count()
    )

    duplicate_count = total_records - distinct_records

    quality_results.append({
        "table": table_name,
        "check": f"Duplicate check: {key_column}",
        "failed_records": duplicate_count,
        "status": "PASS" if duplicate_count == 0 else "FAIL"
    })

check_duplicates(
    sales,
    "gold_sales_performance",
    "order_id"
)

check_duplicates(
    customers,
    "gold_customer_360",
    "customer_id"
)

check_duplicates(
    products,
    "gold_product_performance",
    "product_id"
)

check_duplicates(
    sellers,
    "gold_seller_performance",
    "seller_id"
)

print("Uniqueness checks completed successfully!")

Uniqueness checks completed successfully!


In [0]:
from pyspark.sql import functions as F

checks = []

def run_check(table, condition, name):
    df = spark.table(f"workspace.olist_ecommerce.{table}")
    failed = df.filter(condition).count()
    checks.append((table, name, failed, "PASS" if failed == 0 else "FAIL"))

run_check(
    "gold_sales_performance",
    F.col("total_item_value") < 0,
    "Negative item value"
)

run_check(
    "gold_sales_performance",
    F.col("item_count") < 0,
    "Negative item count"
)

run_check(
    "gold_product_performance",
    F.col("units_sold") < 0,
    "Negative units sold"
)

run_check(
    "gold_seller_performance",
    F.col("units_sold") < 0,
    "Negative seller units"
)

run_check(
    "gold_payment_analysis",
    F.col("total_payment_value") < 0,
    "Negative payment value"
)

run_check(
    "gold_review_analysis",
    (F.col("review_score_int") < 1) | (F.col("review_score_int") > 5),
    "Invalid review score"
)

run_check(
    "gold_delivery_performance",
    F.col("avg_delivery_days") < 0,
    "Negative delivery days"
)

business_quality_df = spark.createDataFrame(
    checks,
    ["table", "check", "failed_records", "status"]
)

display(business_quality_df)

print("Business rule validation completed successfully!")

table,check,failed_records,status
gold_sales_performance,Negative item value,0,PASS
gold_sales_performance,Negative item count,0,PASS
gold_product_performance,Negative units sold,0,PASS
gold_seller_performance,Negative seller units,0,PASS
gold_payment_analysis,Negative payment value,0,PASS
gold_review_analysis,Invalid review score,0,PASS
gold_delivery_performance,Negative delivery days,0,PASS


Business rule validation completed successfully!


In [0]:
total_checks = business_quality_df.count()

passed_checks = (
    business_quality_df
    .filter(F.col("status") == "PASS")
    .count()
)

failed_checks = (
    business_quality_df
    .filter(F.col("status") == "FAIL")
    .count()
)

print("DATA QUALITY SUMMARY")
print("Total checks:", total_checks)
print("Passed checks:", passed_checks)
print("Failed checks:", failed_checks)

if failed_checks == 0:
    print("Overall data quality status: PASS")
else:
    print("Overall data quality status: REVIEW REQUIRED")

DATA QUALITY SUMMARY
Total checks: 7
Passed checks: 7
Failed checks: 0
Overall data quality status: PASS
